# Paso 3 · Preprocesamiento y modelado con scikit-learn

**Qué hace este notebook**
1. Carga el dataset completo (solo las columnas necesarias) y aplica el mismo filtro y la misma partición del notebook 2.
2. Preprocesa: imputación, escalado y one-hot, **ajustados solo con el conjunto de entrenamiento**.
3. Entrena los seis modelos con `GridSearchCV` (cv=3, `roc_auc`), mide tiempos y calcula métricas.
4. Guarda en Drive los resultados y las puntuaciones del conjunto de prueba, que se usarán en DeLong, McNemar y bootstrap.

**Puntos de control:** al terminar cada modelo se guarda su resultado. Si Colab se desconecta, vuelve a ejecutar todo el notebook: los modelos ya terminados se saltan solos.

## 1. Configuración y hardware

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, glob, time, json, warnings
import numpy as np, pandas as pd, joblib, psutil
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

RUTA = '/content/drive/MyDrive/ML_LendingClub'   # <- ajusta si tu carpeta se llama distinto
SEMILLA = 42
USAR_HIST = True   # True: HistGradientBoostingClassifier (permitido por el enunciado si el costo es prohibitivo); False: GradientBoostingClassifier
RAPIDO = False     # solo para pruebas: True usa rejillas mínimas. Déjalo en False para la corrida real

DIR_RES = RUTA + '/resultados_sklearn'
os.makedirs(DIR_RES, exist_ok=True)

HW = {'nucleos': os.cpu_count(), 'ram_gb': round(psutil.virtual_memory().total / 1e9, 1)}
print('Hardware declarado:', HW)

Hardware declarado: {'nucleos': 8, 'ram_gb': 54.8}


## 2. Cargar datos, filtrar y unir la partición común
Se cargan **solo las columnas que usaremos** (más rápido y con menos memoria). Se aplica el mismo filtro del notebook 2 y se asigna `train`/`test` leyendo el archivo de partición por `id`.

In [3]:
NUM = ['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 'fico_range_high', 'delinq_2yrs',
       'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc',
       'pub_rec_bankruptcies']
CAT = ['purpose', 'home_ownership', 'addr_state', 'verification_status']
USECOLS = ['id', 'loan_status', 'term', 'emp_length'] + NUM + CAT

candidatos = [f for f in glob.glob(RUTA + '/**/accepted_2007_to_2018Q4.csv', recursive=True) if os.path.isfile(f)]
if not candidatos:
    candidatos = sorted(f for f in glob.glob(RUTA + '/*accepted*') if os.path.isfile(f))
archivo = candidatos[0]
print('Archivo:', archivo)

cols_archivo = pd.read_csv(archivo, nrows=0).columns
faltan = [c for c in USECOLS if c not in cols_archivo]
assert not faltan, f'Columnas que no existen en el archivo: {faltan}'

t0 = time.time()
df = pd.read_csv(archivo, usecols=USECOLS, low_memory=False)
t_carga = time.time() - t0

df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].reset_index(drop=True)
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)
df['id'] = df['id'].astype(str)

particion = pd.read_parquet(RUTA + '/split_80_20_seed42.parquet')
df['split'] = df['id'].map(particion.set_index('id')['split'])
assert df['split'].notna().all(), 'Hay filas sin asignación train/test'

print(f'Carga: {t_carga:,.1f} s | filas: {len(df):,}')
print(df['split'].value_counts())

Archivo: /content/drive/MyDrive/ML_LendingClub/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv
Carga: 20.8 s | filas: 1,345,310
split
train    1076248
test      269062
Name: count, dtype: int64


## 3. Preprocesamiento (ajustado solo con train)
- `term` y `emp_length` vienen como texto (`' 36 months'`, `'10+ years'`): se convierten a números.
- Numéricas: imputación con la **mediana de train** y escalado estándar.
- Categóricas: one-hot, ignorando categorías no vistas en train.

Todo lo que "aprende" parámetros (medianas, medias, categorías) se ajusta con `fit` **solo** en train y se aplica a test con `transform`. Así el conjunto de prueba no influye en el entrenamiento.

In [4]:
t0 = time.time()

df['term_meses'] = df['term'].str.extract(r'(\d+)')[0].astype(float)
df['emp_length_anios'] = df['emp_length'].str.extract(r'(\d+)')[0].astype(float)
df.loc[df['emp_length'] == '< 1 year', 'emp_length_anios'] = 0.0
for c in CAT:
    df[c] = df[c].fillna('Desconocido')

NUM_COLS = NUM + ['term_meses', 'emp_length_anios']
FEATURES = NUM_COLS + CAT

train = df[df['split'] == 'train']
test = df[df['split'] == 'test']
y_train, y_test = train['default'].values, test['default'].values
ids_test = test['id'].values

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('esc', StandardScaler())]), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32), CAT),
], sparse_threshold=0)

X_train = pre.fit_transform(train[FEATURES]).astype(np.float32)
X_test = pre.transform(test[FEATURES]).astype(np.float32)
t_prep = time.time() - t0

nombres_features = list(pre.get_feature_names_out())
joblib.dump({'pre': pre, 'features': FEATURES, 'nombres': nombres_features}, DIR_RES + '/preprocesador.joblib')

print(f'Preprocesamiento: {t_prep:,.1f} s')
print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('Tasa de default  train: %.3f %%  |  test: %.3f %%' % (y_train.mean() * 100, y_test.mean() * 100))

Preprocesamiento: 7.8 s
X_train: (1076248, 91) | X_test: (269062, 91)
Tasa de default  train: 19.963 %  |  test: 19.963 %


## 4. Definición de modelos y espacios de búsqueda
- **Regresión logística (regularización L2 por defecto) y LinearSVC:** `C = 1 / (regParam × n)` con `regParam ∈ {1e-6, 1e-5, 1e-4}` y `n` = tamaño de train. Es una equivalencia aproximada con Spark (hay que decirlo en el informe).
- **LinearSVC** se probó primero con `loss="hinge"` para coincidir con la pérdida de Spark, pero esa configuración dio una solución degenerada (AUC 0.515). Tras el diagnóstico de la sección final, el modelo definitivo usa `loss="squared_hinge"` con `dual=False` (AUC 0.709).
- **Gradient boosting:** con `USAR_HIST=True` se usa `HistGradientBoostingClassifier`, sustitución que el enunciado permite si el costo es prohibitivo (hay que justificarla). Se desactiva `early_stopping` para que el número de iteraciones sea el de la rejilla, igual que `maxIter` en Spark.
- **Naive Bayes** gaussiano, sin búsqueda de hiperparámetros.

In [5]:
n_train = len(y_train)
Cs = [1 / (r * n_train) for r in (1e-6, 1e-5, 1e-4)]

def g(normal, rapido):
    return rapido if RAPIDO else normal

if USAR_HIST:
    gb = HistGradientBoostingClassifier(learning_rate=0.1, early_stopping=False, random_state=SEMILLA)
    grid_gb = g({'max_iter': [50, 100], 'max_depth': [3, 5]}, {'max_iter': [5], 'max_depth': [3]})
else:
    gb = GradientBoostingClassifier(learning_rate=0.1, random_state=SEMILLA)
    grid_gb = g({'n_estimators': [50, 100], 'max_depth': [3, 5]}, {'n_estimators': [5], 'max_depth': [3]})

MODELOS = {   # nombre: (estimador, rejilla o None)   -- orden: de lo más rápido a lo más lento
    'NaiveBayes':         (GaussianNB(), None),
    'LogisticRegression': (LogisticRegression(solver='lbfgs', max_iter=300),
                           g({'C': Cs}, {'C': Cs[:1]})),
    'DecisionTree':       (DecisionTreeClassifier(random_state=SEMILLA),
                           g({'max_depth': [5, 10, 15]}, {'max_depth': [3]})),
    'LinearSVC':          (LinearSVC(loss='hinge', dual=True, max_iter=2000, random_state=SEMILLA),
                           g({'C': Cs}, {'C': Cs[:1]})),
    'GradientBoosting':   (gb, grid_gb),
    'RandomForest':       (RandomForestClassifier(random_state=SEMILLA),
                           g({'n_estimators': [10, 50, 100], 'max_depth': [5, 10, 15]},
                             {'n_estimators': [5], 'max_depth': [3]})),
}
print('Valores de C (regParam 1e-6, 1e-5, 1e-4):', [round(c, 5) for c in Cs])
print('Modelos:', list(MODELOS))

Valores de C (regParam 1e-6, 1e-5, 1e-4): [0.92915, 0.09292, 0.00929]
Modelos: ['NaiveBayes', 'LogisticRegression', 'DecisionTree', 'LinearSVC', 'GradientBoosting', 'RandomForest']


## 5. Entrenar, evaluar y guardar (con puntos de control)
Para cada modelo:
- `GridSearchCV(cv=3, scoring='roc_auc', n_jobs=-1)` sobre la rejilla (sin `Pipeline`, como pide el enunciado).
- **Tiempo de entrenamiento** = búsqueda con validación cruzada + reajuste final. **Tiempo de predicción** aparte.
- Puntuaciones del conjunto de prueba: `predict_proba[:, 1]`, o `decision_function` en LinearSVC. El AUC solo depende del orden de las puntuaciones.
- Métricas con umbral 0,5 en probabilidades y 0 en `decision_function`.

Los modelos lentos (gradient boosting y bosque aleatorio) quedan al final, así los resultados rápidos aparecen primero.

In [6]:
def a_nativo(v):
    return v.item() if hasattr(v, 'item') else v

def correr_modelo(nombre, estimador, rejilla):
    ruta_json = f'{DIR_RES}/{nombre}.json'
    if os.path.exists(ruta_json):
        print(f'[{nombre}] ya estaba terminado, se salta.')
        return
    print(f'[{nombre}] entrenando...')
    t0 = time.time()
    if rejilla:
        busqueda = GridSearchCV(estimador, rejilla, cv=3, scoring='roc_auc', n_jobs=-1, refit=True)
        busqueda.fit(X_train, y_train)
        modelo, mejores, auc_cv = busqueda.best_estimator_, busqueda.best_params_, busqueda.best_score_
    else:
        modelo = estimador.fit(X_train, y_train)
        mejores, auc_cv = {}, None
    t_fit = time.time() - t0

    t0 = time.time()
    if hasattr(modelo, 'predict_proba'):
        score, umbral = modelo.predict_proba(X_test)[:, 1], 0.5
    else:
        score, umbral = modelo.decision_function(X_test), 0.0
    t_pred = time.time() - t0

    y_pred = (score >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    res = {
        'modelo': nombre,
        'mejores_params': {k: a_nativo(v) for k, v in mejores.items()},
        'auc_cv': None if auc_cv is None else float(auc_cv),
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'precision': float(precision_score(y_test, y_pred, zero_division=0)),
        'recall': float(recall_score(y_test, y_pred)),
        'f1': float(f1_score(y_test, y_pred)),
        'auc': float(roc_auc_score(y_test, score)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'umbral': umbral,
        't_entrenamiento_s': round(t_fit, 1), 't_prediccion_s': round(t_pred, 2),
        't_preprocesamiento_s': round(t_prep, 1), 't_carga_s': round(t_carga, 1),
        'hardware': HW,
    }
    pd.DataFrame({'id': ids_test, 'default': y_test, 'score': score}).to_parquet(f'{DIR_RES}/scores_{nombre}.parquet', index=False)
    joblib.dump(modelo, f'{DIR_RES}/{nombre}.joblib')
    with open(ruta_json, 'w') as f:      # el json se escribe al final: es la marca de "terminado"
        json.dump(res, f, indent=1)
    print(f"[{nombre}] listo en {t_fit/60:.1f} min | AUC={res['auc']:.4f} | mejores={res['mejores_params']}")

for nombre, (est, rej) in MODELOS.items():
    correr_modelo(nombre, est, rej)

[NaiveBayes] entrenando...
[NaiveBayes] listo en 0.0 min | AUC=0.6293 | mejores={}
[LogisticRegression] entrenando...
[LogisticRegression] listo en 0.5 min | AUC=0.7096 | mejores={'C': 0.009291538753149831}
[DecisionTree] entrenando...
[DecisionTree] listo en 0.9 min | AUC=0.7019 | mejores={'max_depth': 10}
[LinearSVC] entrenando...
[LinearSVC] listo en 31.6 min | AUC=0.5152 | mejores={'C': 0.9291538753149833}
[GradientBoosting] entrenando...
[GradientBoosting] listo en 2.0 min | AUC=0.7179 | mejores={'max_depth': 5, 'max_iter': 100}
[RandomForest] entrenando...
[RandomForest] listo en 10.8 min | AUC=0.7143 | mejores={'max_depth': 15, 'n_estimators': 100}


## 6. Consolidar resultados
Tabla de métricas y tiempos por modelo, y un único archivo con las puntuaciones de todos los modelos (para DeLong, McNemar y bootstrap).

In [11]:
filas = []
for nombre in MODELOS:
    p = f'{DIR_RES}/{nombre}.json'
    if os.path.exists(p):
        filas.append(json.load(open(p)))
resultados = pd.DataFrame(filas).set_index('modelo')
cols = ['auc', 'auc_cv', 'accuracy', 'precision', 'recall', 'f1', 't_entrenamiento_s', 't_prediccion_s', 'mejores_params']
resultados[cols]

,auc,auc_cv,accuracy,precision,recall,f1,t_entrenamiento_s,t_prediccion_s,mejores_params
modelo,,,,,,,,,
NaiveBayes,0.629350,NaN,0.579487,0.270519,0.652182,0.382415,0.5,0.29,{}
LogisticRegression,0.709610,0.708254,0.801496,0.518299,0.079628,0.138048,32.9,0.06,{'C': 0.009291538753149831}
DecisionTree,0.701928,0.695600,0.800581,0.503751,0.070003,0.122924,53.0,0.03,{'max_depth': 10}
LinearSVC,0.709246,0.707975,0.801696,0.549776,0.036603,0.068636,72.0,0.05,{'C': 0.09291538753149832}
GradientBoosting,0.717860,0.714420,0.803647,0.571359,0.065665,0.117792,117.4,1.79,"{'max_depth': 5, 'max_iter': 100}"
RandomForest,0.714277,0.711510,0.802005,0.613790,0.022043,0.042559,646.2,3.01,"{'max_depth': 15, 'n_estimators': 100}"


In [12]:
scores = None
for nombre in MODELOS:
    p = f'{DIR_RES}/scores_{nombre}.parquet'
    if os.path.exists(p):
        s = pd.read_parquet(p).rename(columns={'score': nombre})
        scores = s if scores is None else scores.merge(s[['id', nombre]], on='id')
scores.to_parquet(RUTA + '/scores_sklearn.parquet', index=False)
resultados[cols].to_csv(RUTA + '/resultados_sklearn.csv')
print('Guardado:', RUTA + '/scores_sklearn.parquet', scores.shape)
print('Matrices de confusión (tn, fp, fn, tp):')
print(resultados[['tn', 'fp', 'fn', 'tp']])

Guardado: /content/drive/MyDrive/ML_LendingClub/scores_sklearn.parquet (269062, 8)
Matrices de confusión (tn, fp, fn, tp):
                        tn     fp     fn     tp
modelo                                         
NaiveBayes          120888  94462  18682  35030
LogisticRegression  211375   3975  49435   4277
DecisionTree        211646   3704  49952   3760
LinearSVC           213740   1610  51746   1966
GradientBoosting    212704   2646  50185   3527
RandomForest        214605    745  52528   1184


In [9]:
import time, warnings, numpy as np
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# submuestra SOLO para diagnóstico (no es el modelo final)
Xs, _, ys, _ = train_test_split(X_train, y_train, train_size=200_000,
                                stratify=y_train, random_state=42)
Xv, ys_v = X_test, y_test   # evaluación sobre el test completo

pruebas = {
 'hinge dual  C=0.01 it=2000':  dict(loss='hinge', dual=True,  C=0.01, max_iter=2000),
 'hinge dual  C=0.01 it=20000': dict(loss='hinge', dual=True,  C=0.01, max_iter=20000),
 'sqhinge primal C=0.01':       dict(loss='squared_hinge', dual=False, C=0.01, max_iter=2000),
 'sqhinge primal C=0.001':      dict(loss='squared_hinge', dual=False, C=0.001, max_iter=2000),
}
for nombre, kw in pruebas.items():
    t = time.time()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        m = LinearSVC(random_state=42, **kw).fit(Xs, ys)
    auc = roc_auc_score(ys_v, m.decision_function(Xv))
    print(f'{nombre:30s} AUC={auc:.4f}  n_iter={m.n_iter_}  '
          f'warnings={len(w)}  {time.time()-t:.0f}s')

hinge dual  C=0.01 it=2000     AUC=0.5778  n_iter=94  warnings=0  7s
hinge dual  C=0.01 it=20000    AUC=0.5778  n_iter=94  warnings=1  7s
sqhinge primal C=0.01          AUC=0.7088  n_iter=5  warnings=1  4s
sqhinge primal C=0.001         AUC=0.7089  n_iter=5  warnings=1  2s


In [10]:
import glob, os, shutil
# 1) conservar la evidencia del SVM con hinge (no se borra)
os.makedirs(DIR_RES + '/_hinge_descartado', exist_ok=True)
for f in glob.glob(f'{DIR_RES}/LinearSVC*') + glob.glob(f'{DIR_RES}/scores_LinearSVC*'):
    shutil.move(f, DIR_RES + '/_hinge_descartado/')

# 2) nuevo SVM (squared_hinge, primal) con la misma rejilla de C
MODELOS['LinearSVC'] = (LinearSVC(loss='squared_hinge', dual=False, max_iter=2000,
                                  random_state=SEMILLA),
                        {'C': Cs})
correr_modelo('LinearSVC', *MODELOS['LinearSVC'])

[LinearSVC] entrenando...
[LinearSVC] listo en 1.2 min | AUC=0.7092 | mejores={'C': 0.09291538753149832}
